## Supervised Learning Assessment 2


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats


from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import Lasso, Ridge, ElasticNet, LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GroupKFold, GridSearchCV, cross_val_score
from sklearn.metrics import mean_squared_error, r2_score

import warnings
warnings.filterwarnings('ignore')

# Reproducibility
np.random.seed(270)


---
## Part 1: Exploratory Data Analysis
### 1.1 Load and Inspect Data
We load both datasets upfront. Key structural observations:
- Training: 4000 samples (rows) * 42 columns (y, group_id, X1-X40)
- Test: 1000 samples (rows) * 41 columns (obs_id, X1-X40). No y and no group_id
- The test set comes from laboratories that are not present in training, so our model must generalise across batch effects.


In [ ]:
# Load data
train = pd.read_csv('train1.csv')
test = pd.read_csv('test_public.csv')


features = [f'X{i}' for i in range(1, 41)]

print("Structure of the datasets:")
print(f"features: {features}")
print(f"Training:  {train.shape[0]} rows * {train.shape[1]} columns")
print(f"Test:      {test.shape[0]} rows * {test.shape[1]} columns")
print(f"\nTraining columns: {list(train.columns)}")
print(f"Test columns:     {list(test.columns[:5])}")
print(f"\nUnique groups: {train['group_id'].nunique()} unique")
print(f"Samples per group: {train.groupby('group_id').size().unique()[0]}")
print(f"\nMissing values in train: {train.isnull().sum().sum()}, test: {test.isnull().sum().sum()}")


Our main interesting takeaway here is that there are no missing values, so we do not need to perform any preprocessing for this.


### 1.2 Response Variable
We check the distribution of y to see whether transformations (e.g. log) are needed and what loss functions are appropriate.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3))

# Histogram
axes[0].hist(train['y'], bins=40, edgecolor='black', alpha=0.7, color='steelblue')
axes[0].axvline(train['y'].mean(), color='red', ls='--', label=f"Mean = {train['y'].mean():.2f}")
axes[0].axvline(train['y'].median(), color='orange', ls='--', label=f"Median = {train['y'].median():.2f}")
axes[0].set_xlabel('y')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Response Variable y')
axes[0].legend()

# QQ plot
stats.probplot(train['y'], plot=axes[1])
axes[1].set_title('Q-Q Plot of y')
plt.tight_layout()
plt.show()


We find that the distribution is approximately symmetric and close to gaussian, indicating that no transformation is needed for y


### 1.3 Feature Distributions and Skewness
We check the distribution of our features. From research, gene expression data can often be right-skewed, and so may require transformation.


We find that all features in our dataset have a range of [0,1] indicating they may already be pre-processed or transformed. They also exhibit near-zero skewness.
If we choose to use regularised models, like Lasso which applies a penalty to coefficients, we  may not need any scaling. As all features share a common range, gross differences are absent here. However, we may still find residual variance differences, and so, standardScalar may be useful. It is standard practice, and would be harmless even if useless in this scenario. 
Log-transformations are unneccessary due to the absence of skewness.


### 1.4 Batch Effects
Since each laboratory uses different equipment/protocols, measurements from the same lab may share systematic biases, termed batch effects by the assignment brief. TThis can lead to misleading effects, particularly in validation.
- If batch effects are large, standard K-Fold CV as a validation could be overly optimistic as training and validation share labs.
- The test set comes from unseen labs, so we should try to generalise across batch effects.


In [ ]:
# How much does y vary BETWEEN labs vs WITHIN labs?
group_stats = train.groupby('group_id')['y'].agg(['mean', 'std'])

print(f"Overall y descriptors:  mean = {train['y'].mean():.2f}, std = {train['y'].std():.2f}")
print("lab means descriptors:")
print(f"range = [{group_stats['mean'].min():.2f}, {group_stats['mean'].max():.2f}]")
print(f"Std of lab means: {group_stats['mean'].std():.2f}")
print(f"Mean of within-lab stds: {group_stats['std'].mean():.2f}")

print(f"\nRatio (between-lab var / total var): {group_stats['mean'].var() / train['y'].var():.2f}")


In [ ]:
# Visualise
fig, ax = plt.subplots(figsize=(12, 3))
group_order = group_stats['mean'].sort_values().index
bp = ax.boxplot([train[train['group_id']==g]['y'].values for g in group_order],
                patch_artist=True, showfliers=False,
                medianprops=dict(color='red', linewidth=1.5))
for patch in bp['boxes']:
    patch.set_facecolor('steelblue')
    patch.set_alpha(0.6)
ax.set_xlabel('Laboratory (sorted by mean y)')
ax.set_ylabel('y')
ax.set_title('Batch Effects: Response y by Laboratory')
ax.set_xticks(range(1, 41))
ax.set_xticklabels([str(g) for g in group_order], rotation=90, fontsize=7)
plt.tight_layout()
plt.show()


We find a couple of interesting takeaways. 
Our lab means range of [6.25, 23.03] indicates that the average y value differs greatly across labs.
83% of the variation seen in y can be determined from the lab sample it came from. Only 17% is within lab variation. This poses a problem for Standard K-Fold.
As Standard K-Fold splits observations randomly, observations for one lab may land in both the training and validation fold. The model would be able to correctly predict the structure of one lab, but may not learn the actual biology of the dataset - i.e. be able to predict the gene-expression within labs.
This would lead to a model performing well within validation, but only learning which lab a given sample belongs to, rather than the gene expression and response relationships. 
We choose to mitigate these effects by using GroupKFold for our validation method instead. This ensures that the observations in a given lab are either places in training or validation only, and not split across both, allowing us to generalise these batch effects across labs, and leading to a more accurate cross-validated MSE.


### 1.5 Feature-Response Correlations
We further evaluate the structure of our dataset, identifying which genes are most linearly associated with y.
This will help us select features for our models, as well as whether a spare or dense model is more appropriate.


We find that the Signal is concentrated in around 6 genes (X5, X6, X29, X19, X36, X13).
We also find that around 25 genes have negligible correlation with y. If we choose to use Lasso, it should disregard these features.


### 1.6 Multicollinearity Check
In multiple linear regression, we estimate coeficients using OLS. High correlation between features can cause inflated variance, and so OLS coefficients become unreliable and highly sensitive to small changes in the data. We check for multicollinearity and see if our OLS estimates may affect the accuracy of our predictors.


Key takeaways here:
- No severe multicollinearity (i.e. no pairs above |r| = 0.7).
- Top predictors (X5, X6, X29) are not strongly correlated with each other.
Multicollinearity is moderate here. Lasso would not be impacted greatly (tendency to arbitrarily pick correlated pairs). Ridges L2 penalty means multicollinearity is also not an issue here.


### 1.7 PCA: Dimensionality and Laboratory Clustering
PCA is performed to gain further insight into the structure of our data. We are looking for two different things:
1. Whether the data lies on a low-dimensional subspace, which may indicate that PCR with few components could work well.
2. Identifying if our laboratories cluster in PC space, which would confirm batch effects exist in the feature space, not just in y.


In [ ]:
scaler_eda = StandardScaler()
X_scaled_eda = scaler_eda.fit_transform(train[features])

# Full PCA
pca_full = PCA().fit(X_scaled_eda)
pve = pca_full.explained_variance_ratio_
cum_pve = np.cumsum(pve)
n90 = np.argmax(cum_pve >= 0.9) + 1
n95 = np.argmax(cum_pve >= 0.95) + 1

fig, axes = plt.subplots(1, 2, figsize=(12, 3))

# Scree plot
axes[0].bar(range(1, 41), pve, alpha=0.6, color='steelblue', label='Individual PVE')
axes[0].plot(range(1, 41), cum_pve, 'ro-', markersize=4, label='Cumulative PVE')
axes[0].axhline(0.9, ls='--', color='grey', alpha=0.5)
axes[0].set_xlabel('Principal Component')
axes[0].set_ylabel('Proportion of Variance Explained')
axes[0].set_title('Scree Plot')
axes[0].legend()
axes[0].annotate(f'{n90} PCs for 90%', xy=(n90, 0.9), fontsize=10, color='red')

# PCA scatter coloured by lab
pca_2d = PCA(n_components=2)
pcs = pca_2d.fit_transform(X_scaled_eda)
scatter = axes[1].scatter(pcs[:, 0], pcs[:, 1], c=train['group_id'], cmap='tab20', alpha=0.4, s=10)
axes[1].set_xlabel(f'PC1 ({pca_2d.explained_variance_ratio_[0]*100:.1f}% variance)')
axes[1].set_ylabel(f'PC2 ({pca_2d.explained_variance_ratio_[1]*100:.1f}% variance)')
axes[1].set_title('PCA: Samples Coloured by Laboratory')

plt.tight_layout()
plt.show()


The scree plot shows no clear elbow, with 24 components needed for 90% variance.This means the data has no useful low-dimensional structure, which makes PCR a poor choice here. 
The PCA scatter shows labs do not cluster in feature space, confirming batch effects live in y and not the feature space, which may reduce the severity of these batch effects and the generalisation problem.
Our variance structure has a large spread, indicating favour for Lasso over PCR, as there is no clustering of labs.


---
## Part 2: Model Development and Interpretation


In [ ]:
# Prepare data
X_train = train[features].values
y_train = train['y'].values
groups = train['group_id'].values
X_test = test[features].values

# Each fold holds out ~4 entire laboratories (4 × 100 = 400 samples)
gkf = GroupKFold(n_splits=10)

# Verify fold structure
for i, (tr_idx, val_idx) in enumerate(gkf.split(X_train, y_train, groups)):
    val_labs = sorted(np.unique(groups[val_idx]))
    if i < 3:
        print(f"Fold {i+1}: Train {len(tr_idx)} samples | Validate {len(val_idx)} samples | Held-out labs: {val_labs}")

results = {}


n_splits=10 holds out exactly 4 labs per fold, giving 400 validation samples, which is large enough for a stable MSE estimate while keeping 3600 training samples per fold. 
GroupKFold is used instead of standard KFold because the 83% between-lab variance shown in the EDA section.
 Passing groups=train['group_id'].values mnakes sure that all observations from the same laboratory stay together, preventing leakage back to standard KFold behaviour.


### 2.1 Model 1: Lasso Regression (L1 Penalty)
The EDA showed signal concentrated in ~6 genes. Lasso's L1 penalty shrinks irrelevant coefficients exactly to zero, performing automatic variable selection. From our EDA, this makes Lasso a good potential model.
It can provide us with:
- A sparse, interpretable model, signalling which genes matter for responses.
- Protection against overfitting, as we have many other features.
$\alpha$ controls regularisation strength, i.e. larger $\alpha$ mean that more coefficients are discarded.
We select the value of $\alpha$ based on searching through 50 values on a log-scale from $10^{-4}$ to 10.


In [ ]:
lasso_alphas = np.logspace(-4, 1, 50)

lasso_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('lasso', Lasso(max_iter=10000)) #to avoid convergence warnings for small alphas
])

lasso_grid = GridSearchCV(
    lasso_pipe,
    param_grid={'lasso__alpha': lasso_alphas},
    cv=gkf,
    scoring='neg_mean_squared_error',
    n_jobs=-1,
    return_train_score=True
)
lasso_grid.fit(X_train, y_train, groups=groups)

# Extract results
lasso_cv_mse = -lasso_grid.cv_results_['mean_test_score']
lasso_cv_std = lasso_grid.cv_results_['std_test_score']
lasso_train_mse = -lasso_grid.cv_results_['mean_train_score']

best_lasso_alpha = lasso_grid.best_params_['lasso__alpha']
best_lasso_mse = -lasso_grid.best_score_
best_lasso_std = lasso_cv_std[lasso_grid.best_index_]

print(f"Best alpha: {best_lasso_alpha:.6f}")
print(f"CV MSE:     {best_lasso_mse:.4f} ± {best_lasso_std:.4f}")

# Refit at best alpha to inspect coefficients
lasso_final = Pipeline([('scaler', StandardScaler()), ('lasso', Lasso(alpha=best_lasso_alpha, max_iter=10000))])
lasso_final.fit(X_train, y_train)
lasso_coefs = pd.Series(lasso_final.named_steps['lasso'].coef_, index=features)

print(f"\nNon-zero coefficients: {(lasso_coefs != 0).sum()}/40")
print("\nSelected genes (sorted by |coefficient|):")
print(lasso_coefs[lasso_coefs != 0].sort_values(key=abs, ascending=False).round(4))

results['Lasso'] = {'mse': best_lasso_mse, 'std': best_lasso_std, 'model': lasso_final}


### 2.2 Model 2: Gradient Boosting
Our EDA showed that Lasso is a natural choice, by suggesting the model is sparse due to only a few predictive features. 
We compliment this with gradient boost, as our 83% between-lab variance, and a failure of linear structures explaining y could mean that non-linear structures are present. Our PCA seems to support this hypothesis: labs did not cluster in the 2D PCA space, and variance was widely spread out, so they may cluster in higher dimensions. 
Gradient boosting builds trees sequentially, and so may be able to capture these non-linear patterns better. So while both models are wildly different, they allow us to approach the dataset from two different directions.


In [ ]:
gb_pipe = Pipeline([
    ('gb', GradientBoostingRegressor(random_state=270))
])

gb_grid = GridSearchCV(
    gb_pipe,
    param_grid={
        'gb__n_estimators': [100, 300],
        'gb__learning_rate': [0.05, 0.1],
        'gb__max_depth': [3, 5],
        'gb__min_samples_leaf': [10, 20],
        'gb__subsample': [0.8]
    },
    cv=gkf,
    scoring='neg_mean_squared_error',
    n_jobs=-1,
    return_train_score=True
)
gb_grid.fit(X_train, y_train, groups=groups)

best_gb_mse = -gb_grid.best_score_
best_gb_std = gb_grid.cv_results_['std_test_score'][gb_grid.best_index_]

print(f"Best params: {gb_grid.best_params_}")
print(f"CV MSE:      {best_gb_mse:.4f} ± {best_gb_std:.4f}")

# Feature importance
gb_final = gb_grid.best_estimator_
gb_final.fit(X_train, y_train)
gb_importance = pd.Series(gb_final.named_steps['gb'].feature_importances_, index=features)
print("\nTop 10 features by importance:")
print(gb_importance.sort_values(ascending=False).head(10).round(4))

results['Gradient Boosting'] = {'mse': best_gb_mse, 'std': best_gb_std, 'model': gb_final}


#### Parameter justification
n_estimators: [100, 300]:
- Controls the number of sequential trees. 100 and 300 bracket the typical range for a dataset of this size, we also tried 500 but removed it after excessive runtime for diminishing results.
learning_rate: [0.05, 0.1]:
- Each tree's contribution to the final prediction is multiplied by this value before the next tree is fitted. A smaller learning rate means each tree corrects errors more cautiously, requiring more trees to converge but generally producing a better regularised model.
max_depth: [3, 5]:
- Controls how many feature interactions each tree can capture. Depth 3 allows three-way interactions (e.g. X5 * X6 * X29). Depth 5 allows deeper interactions but risks overfitting.
min_samples_leaf: [10, 20]:
- Sets the minimum number of observations required in each last node of a tree. Higher values prevent the model from making splits that represent only a handful of lab-specific observations, which then acts as a regulariser against overfitting. We choose conservative values here, as we only have roughly 100 obs per lab.
subsample: [0.8]:
- tree is fitted on a random 80% of the training data rather than the full dataset, introducing randomness that reduces overfitting. Fixed at 0.8 rather than searched, leading to a minimal cost to bias, while cutting down on processing time.


### 2.3 Sanity checks for very low MSE
Much lower than the 4.54 floor in EDA, this is very suspicious, so some checks here are made to verify if this result is due to a leakage error.


Our checks here show us:
- GroupKFold was correctly implemented: group_id excluded from features, dimensions verified (4000×40), y not standardised, and manual fold-by-fold verification confirmed the CV MSE of 1.04 is genuine and not a GridSearchCV artefact.
- The MSE of 1.04 violates the theoretical within-lab variance floor of 4.54. This could be overfitting, or finding non-linear feature interactions that can identify labs in high-dimensional spaces.


### 2.4 Model Interpretation: Coefficient and Feature Importance Comparison


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# 1. Lasso coefficients
ax = axes[0]
nz = lasso_coefs[lasso_coefs != 0].sort_values()
ax.barh(nz.index, nz.values, color=['steelblue' if v > 0 else 'salmon' for v in nz])
ax.set_xlabel('Coefficient')
ax.set_title(f'Lasso: {len(nz)}/40 features selected')
ax.axvline(0, color='black', lw=0.5)

# 2. Gradient Boosting importance
ax = axes[1]
gb_imp_sorted = gb_importance.sort_values(ascending=True)
ax.barh(gb_imp_sorted.index, gb_imp_sorted.values, color='steelblue')
ax.set_xlabel('Importance')
ax.set_title('Gradient Boosting: Feature Importance')

plt.tight_layout()
plt.show()

# Cross-method agreement
print("Top 5 Features:")
print(f"Lasso:     {list(lasso_coefs.abs().sort_values(ascending=False).head(5).index)}")
print(f"GB:        {list(gb_importance.sort_values(ascending=False).head(5).index)}")


Both models select the correct features, verifying each other and reassuring our choices.


### 2.5 Model discussion
Although both models agree, we find that GB has wildly lower MSE scores, which is very suspicious behaviour. 
We suspect this behavior stems from potentially two points: 
1. Higher-dimension clustering: In higher dimensions, there may be clustering available that GB is able to exploit over Lasso.
2. GB is able to capture non-linearity, which could decode the batch effects here, and potentially correct for them. For example, if X6 > 0.55 AND X29 < 0.48 AND X37 > 0.52, then this looks like a high-y lab, so predict higher, regardless of clustering.
Point 2 raises an interesting problem however, if GB is doing this, then when presented with test data that GB has not been trained to correct, it may lead to inconsistent predictions, and potentially high stds.


---
## Part 3: Validation and Model Selection
From the assignment brief, test set contains samples from laboratories not present in training. Standard K-Fold CV randomly splits samples, so a training fold and validation fold would share samples from the same lab. Because labs have strong batch effects (Section 1.4), this would:
- Leak lab-specific information into the validation set
- Produce unrealistically low MSE estimates
- Select models that overfit to lab-specific patterns
GroupKFold solves this by ensuring entire laboratories are held out per fold, directly mimicking the test scenario.
### 3.2 Tuning Curves


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Lasso CV path
ax = axes[0]
ax.semilogx(lasso_alphas, lasso_cv_mse, 'b-o', ms=3, label='CV MSE')
ax.fill_between(lasso_alphas, lasso_cv_mse - lasso_cv_std, lasso_cv_mse + lasso_cv_std, alpha=0.2)
ax.semilogx(lasso_alphas, lasso_train_mse, 'g--', alpha=0.6, label='Train MSE')
ax.axvline(best_lasso_alpha, color='r', ls='--', label=f'Best alpha={best_lasso_alpha:.4f}')
ax.set_xlabel('alpha (regularisation)')
ax.set_ylabel('MSE')
ax.set_title('Lasso: GroupKFold CV Path')
ax.legend(fontsize=8)

# Summary comparison bar chart
ax = axes[1]
model_names = list(results.keys())
mses = [results[m]['mse'] for m in model_names]
stds = [results[m]['std'] for m in model_names]
colors = ['steelblue'] * len(model_names)
best_idx = np.argmin(mses)
colors[best_idx] = 'orange'
ax.barh(model_names, mses, xerr=stds, color=colors, edgecolor='black', capsize=3)
ax.set_xlabel('GroupKFold CV MSE (lower is better)')
ax.set_title('Model Comparison')
ax.axvline(min(mses), color='red', ls='--', alpha=0.5)

plt.tight_layout()
plt.show()


In [ ]:
print("-" * 70)
print(f"{'Model':<22} {'CV MSE':>10} {'± Std':>10} {'CV RMSE':>10} {'CV R^2':>10}")
print("-" * 70)

total_var = np.var(y_train)
for name in results:
    mse = results[name]['mse']
    std = results[name]['std']
    rmse = np.sqrt(mse)
    r2_cv = 1 - mse / total_var
    print(f"{name:<22} {mse:>10.4f} {std:>10.4f} {rmse:>10.4f} {r2_cv:>10.4f}")
    
print("-" * 70)
best_model_name = min(results, key=lambda m: results[m]['mse'])
print(f"\nBest model: {best_model_name} (lowest GroupKFold CV MSE)")
print(f"Estimated generalisation MSE: {results[best_model_name]['mse']:.4f}")
print(f"Estimated generalisation RMSE: {np.sqrt(results[best_model_name]['mse']):.4f}")
print(f"Estimated generalisation R^2: {1 - results[best_model_name]['mse']/total_var:.4f}")

# Also compute training metrics for overfitting analysis
print(f"\n{'Model':<22} {'Train MSE':>10} {'Train R^2':>10} {'CV MSE':>10} {'Gap':>10}")
print("-" * 70)
for name in results:
    model = results[name]['model']
    y_pred_train = model.predict(X_train)
    tr_mse = mean_squared_error(y_train, y_pred_train)
    tr_r2 = r2_score(y_train, y_pred_train)
    cv_mse = results[name]['mse']
    gap = cv_mse - tr_mse
    print(f"{name:<22} {tr_mse:>10.4f} {tr_r2:>10.4f} {cv_mse:>10.4f} {gap:>10.4f}")


Gradient Boosting achieves CV MSE = 0.94 (R^2 = 0.96) compared to Lasso's 5.78 (R^2 = 0.77). 
As discussed earlier, this could come from two points: 
1. Higher-dimension clustering: In higher dimensions, there may be clustering available that GB is able to exploit over Lasso.
2. GB is able to capture non-linearity, which could decode the batch effects here, and potentially correct for them. For example, if X6 > 0.55 AND X29 < 0.48 AND X37 > 0.52, then this looks like a high y lab, so predict higher, regardless of clustering.
Investigation could be done through analysing higher PCs, or constructing metrics on variance within labs. None of these were covered in the course materials, so we refrain from trying to explore new investigation methods.
This batch "correction" is learned from 40 training labs, so whether it transfers to new labs is the primary risk. However, the GroupKFold estimate is unbiased and the train-CV gap is modest, so we select GB as the final model.


---
## Part 4: Test Predictions


In [ ]:
best_model_name = min(results, key=lambda m: results[m]['mse'])
final_model = results[best_model_name]['model']

print(f"Selected model: {best_model_name}")
print(f"Estimated generalisation MSE: {results[best_model_name]['mse']:.4f}")

final_model.fit(X_train, y_train)

# Training performance
y_train_pred = final_model.predict(X_train)
train_mse = mean_squared_error(y_train, y_train_pred)
train_r2 = r2_score(y_train, y_train_pred)
print(f"\nTraining MSE: {train_mse:.4f}")
print(f"Training R^2:  {train_r2:.4f}")

y_hat = final_model.predict(X_test)

predictions = pd.DataFrame({
    'obs_id': test['obs_id'],
    'y_hat': y_hat
})

print(f"\nPrediction summary:")
print(f"  Rows:    {len(predictions)}")
print(f"  Columns: {list(predictions.columns)}")
print(f"  y_hat mean: {predictions['y_hat'].mean():.3f} (training y mean: {y_train.mean():.3f})")
print(f"  y_hat std:  {predictions['y_hat'].std():.3f} (training y std: {y_train.std():.3f})")
print(f"  y_hat range: [{predictions['y_hat'].min():.3f}, {predictions['y_hat'].max():.3f}]")

# Save
save_name = "final_predictions.csv"
predictions.to_csv(save_name, index=False)
print(f"\nSaved: {save_name}")
print(predictions.head(10))
